# 第 6 周练习：价格估算器基线 + 误差分析

## 练习目标（理念）

用商品描述文本训练一个**基线价格预测器**，并学会用误差指标评估它：

- **特征**：TF-IDF（把描述变成稀疏词向量）
- **模型**：Ridge 回归（线性、带 L2 正则）
- **技巧**：对价格做 `log1p` 变换再回归，预测时再 `expm1` 还原
- **对比**：先算「永远猜训练集中位数」的傻瓜基线，再看模型是否更好
- **误差分析**：找出绝对误差最大的样本，肉眼看描述里缺了什么信号

## 和本课 Week 6 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 基线（baseline） | 中位数预测 vs TF-IDF+Ridge |
| MAE / RMSE | `mean_absolute_error`、`mean_squared_error` |
| 文本特征 | `TfidfVectorizer`（unigram+bigram） |
| 目标变换 | `TransformedTargetRegressor` + `log1p` / `expm1` |

## 怎么跑

1. 确认 `DATA_PATH` 指向本机的 `human_out.csv`（两列：描述, 价格）
2. 从上到下依次运行每个单元格
3. 对比 Baseline MAE/RMSE 与 Model MAE/RMSE，再读「最差 5 条」做误差分析


In [ ]:
# ========== 依赖（可选）：若环境缺包可取消下一行注释 ==========

# 需要 scikit-learn / numpy；课程虚拟环境通常已装好
# !pip -q install scikit-learn numpy


In [1]:
# ========== 导入：CSV、数值、路径、sklearn 管线 ==========

# 标准库 csv：按行列读人工标注的价格 CSV
import csv
# 标准库 math：开方算 RMSE（Root Mean Squared Error）
import math
# NumPy：中位数等向量化统计
import numpy as np
# pathlib.Path：跨平台写文件路径，避免硬编码字符串拼接
from pathlib import Path
# train_test_split：把样本分成训练集 / 测试集
from sklearn.model_selection import train_test_split
# TfidfVectorizer：文本 → TF-IDF 稀疏特征矩阵
from sklearn.feature_extraction.text import TfidfVectorizer
# Ridge：带 L2 正则的线性回归，文本特征高维时更稳
from sklearn.linear_model import Ridge
# Pipeline：把「向量化 → 回归」串成一步 fit/predict
from sklearn.pipeline import Pipeline
# TransformedTargetRegressor：对 y 做变换（这里 log1p）再训练
from sklearn.compose import TransformedTargetRegressor
# MAE / MSE：回归常用误差指标
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
# ========== 加载数据：两列 CSV → texts 列表 + prices 列表 ==========

# 【注意】这是作者本机绝对路径；换机器时请改成你仓库里 week6/human_out.csv 的实际位置
DATA_PATH = Path('/Users/andela/projects/llm_engineering/week6/human_out.csv')

def load_csv(path: Path):
    # texts：商品描述字符串；prices：对应价格浮点数
    texts = []
    prices = []
    # 文本模式打开 CSV（默认 utf-8 视环境而定）
    with path.open() as f:
        # csv.reader：按逗号分行拆列
        reader = csv.reader(f)
        for row in reader:
            # 列数不足则跳过坏行
            if len(row) < 2:
                continue
            # 第 0 列：描述文本，去掉首尾空白
            text = row[0].strip()
            try:
                # 第 1 列：价格转 float；转失败（表头/脏数据）就跳过
                price = float(row[1])
            except ValueError:
                continue
            texts.append(text)
            prices.append(price)
    return texts, prices

# 读入全部有效样本
texts, prices = load_csv(DATA_PATH)
# 打印条数，快速确认路径没指错
print('Loaded:', len(texts))
# 0 条说明路径或文件格式有问题，直接失败比后面静默算 NaN 更安全
if len(texts) == 0:
    raise ValueError('No data loaded. Check DATA_PATH.')


Loaded: 100


In [3]:
# ========== 划分训练 / 测试：评估必须在「没见过」的样本上 ==========

# test_size=0.2：20% 留作测试；random_state=42：可复现同一划分
X_train, X_test, y_train, y_test = train_test_split(
    texts, prices, test_size=0.2, random_state=42
)


In [4]:
# ========== 傻瓜基线：永远预测「训练集价格中位数」 ==========

# 中位数对极端贵/便宜商品比均值更稳，常作回归基线
median_price = float(np.median(y_train))
# 对测试集每个样本都猜同一个中位数
baseline_preds = [median_price] * len(y_test)
# MAE：平均绝对误差（和价格同单位，好解释）
baseline_mae = mean_absolute_error(y_test, baseline_preds)
# RMSE：均方根误差（对大错更敏感）
baseline_rmse = math.sqrt(mean_squared_error(y_test, baseline_preds))
# 打印基线数字：后面模型必须明显优于这些值才算「有用」
print('Baseline median price:', round(median_price, 2))
print('Baseline MAE:', round(baseline_mae, 2))
print('Baseline RMSE:', round(baseline_rmse, 2))


Baseline median price: 50.0
Baseline MAE: 49.7
Baseline RMSE: 81.66


In [5]:
# ========== 模型：TF-IDF 特征 + 在 log1p(价格) 上做 Ridge ==========

# max_features：词表上限；ngram_range=(1,2)：单词+二元组；min_df=2：至少出现 2 次才进词表
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)
# alpha=1.0：L2 正则强度；越大系数越平滑、越不易过拟合
regressor = Ridge(alpha=1.0)
# 管线顺序：先文本向量化，再 Ridge
pipeline = Pipeline([
    ('tfidf', tfidf),
    ('ridge', regressor)
])
# 价格常右偏：对 y 做 log1p 训练，predict 时用 expm1 还原到美元尺度
model = TransformedTargetRegressor(
    regressor=pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)
# 在训练集上拟合整条管线（含 TF-IDF 词表）
model.fit(X_train, y_train)


,regressor,"Pipeline(step...e', Ridge())])"
,transformer,None
,func,<ufunc 'log1p'>
,inverse_func,<ufunc 'expm1'>
,check_inverse,True
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None


In [6]:
# ========== 评估：在测试集上算 MAE / RMSE，和基线对比 ==========

# predict：内部会走 TF-IDF → Ridge → expm1
preds = model.predict(X_test)
# 模型 MAE：平均差多少钱
mae = mean_absolute_error(y_test, preds)
# 模型 RMSE：大误差会被平方放大
rmse = math.sqrt(mean_squared_error(y_test, preds))
print('Model MAE:', round(mae, 2))
print('Model RMSE:', round(rmse, 2))


Model MAE: 44.12
Model RMSE: 71.64


In [7]:
# ========== 误差分析：打印绝对误差最大的 5 条（看「为什么错」） ==========

def shorten(text, n=180):
    # 描述太长时截断，方便在笔记本里扫读
    return text[:n] + ('...' if len(text) > n else '')

# 收集 (绝对误差, 真实价, 预测价, 原文)
errors = []
for text, actual, pred in zip(X_test, y_test, preds):
    errors.append((abs(actual - pred), actual, pred, text))
# 按绝对误差从大到小排
errors.sort(reverse=True)

# 只看 Top-5 最惨案例：常见原因是品牌/型号/成色等信号在文本里弱或噪声大
for i in range(5):
    err, actual, pred, text = errors[i]
    print('\n---')
    print('Actual:', round(actual, 2), 'Pred:', round(pred, 2), 'AbsErr:', round(err, 2))
    print(shorten(text))



---
Actual: 350.0 Pred: 87.95 AbsErr: 262.05
Title: Dell Optiplex 390 19" Desktop Bundle  
Category: Computer & Peripherals  
Brand: Dell  
Description: All-in-one Dell Optiplex 390 desktop with i3 processor, 4 GB RAM, 1 TB H...

---
Actual: 185.0 Pred: 76.74 AbsErr: 108.26
Title: APS iBoard 6‑inch Aluminum Running Boards  
Category: Automotive Accessories  
Brand: APS  
Description: Heavy‑duty 6‑inch aluminum running boards for Nissan Frontier Crew C...

---
Actual: 120.0 Pred: 39.97 AbsErr: 80.03
Title: Pidoko Kids Train Table – Grey 90‑Piece Train Set  
Category: Toys & Games  
Brand: Pidoko Kids  
Description: A sturdy, large wooden train table with a 90‑piece train set, ...

---
Actual: 120.0 Pred: 53.86 AbsErr: 66.14
Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay...

---
Actual: 110.0 Pred: 50.35 AbsErr: 59.65
Title: HK 13" Heavy Duty Black Push-O

In [8]:
# ========== 快速试玩：丢一条新描述，看模型估多少钱 ==========

def predict_price(description: str) -> float:
    # 单条也要放进列表：sklearn 期望可迭代的样本集合
    return float(model.predict([description])[0])

# 示例商品卡（英文字段保留：训练数据多半是英文描述风格）
sample = ('Title: Example Headphones\n'
          'Category: Electronics\n'
          'Brand: ExampleCo\n'
          'Description: Wireless over-ear headphones with noise cancellation and 30-hour battery life.\n'
          'Details: Includes carrying case, fast USB-C charging, and 2-year warranty.')
print('Predicted price:', round(predict_price(sample), 2))


Predicted price: 48.9
